In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting"
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [5]:
from config import load_config

cfg = load_config(PROJECT_ROOT / "configs/config.yaml")

cfg["data"]["root"] = (
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL"
)

print(cfg["data"])

{'root': '/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL', 'sampling_rate': 100, 'history': 700, 'horizon': 300, 'synthetic': False, 'start': 0, 'layout': 'time_first'}


In [6]:
from data import prepare

cache, manifest = prepare(cfg)

print("Cache:", cache)
print(manifest["splits"])

Cache: data/study_v3/ee7d4f585e459326
{'train': {'records': 339, 'patients': 300, 'windows': 339}, 'val': {'records': 81, 'patients': 75, 'windows': 81}}


In [7]:
from data import ECGDataset
from preprocessing import feature_arrays

arrays, names, details = feature_arrays(
    cfg,
    cache,
    "F0",
    splits=("train", "val"),
)

dataset = ECGDataset(
    cache,
    cfg,
    "train",
    "P0",
    arrays["train"],
)

sample = dataset[0]

print("x:", sample["x"].shape)
print("y:", sample["y"].shape)
print("static:", sample["static"].shape)

x: torch.Size([700, 12])
y: torch.Size([300, 12])
static: torch.Size([0])


In [8]:
from models import build_model
import torch

model = build_model(
    "patchtst",
    static_dim=0,
    horizon=cfg["data"]["horizon"],
    **cfg["model"]
)

x = torch.randn(2, 700, 12)

y = model(x)

print(y.shape)

torch.Size([2, 300, 12])


In [9]:
from train import train_run

run = train_run(
    cfg,
    cache,
    architecture="patchtst",
    preprocessing="P0",
    feature_set="F0",
    epochs=1,
)

print(run)

patchtst_P0_F0_lr0.0005_h64_e1: epoch 1/1; validation patient MAE 0.115753 mV
results/primary_7s3s_v3/pilot/checkpoints/patchtst_P0_F0_lr0.0005_h64_e1


In [10]:
from train import train_run

# Run PatchTST with P0 preprocessing for 60 epochs
run_patchtst_60 = train_run(
    cfg,
    cache,
    architecture="patchtst",
    preprocessing="P0",
    feature_set="F0",
    epochs=60,
    name="patchtst_P0_F0_pilot_60e"
)

print(f"60-Epoch PatchTST Training Completed!")
print(f"Checkpoints saved at: {run_patchtst_60}")

patchtst_P0_F0_pilot_60e: epoch 1/60; validation patient MAE 0.115753 mV
patchtst_P0_F0_pilot_60e: epoch 2/60; validation patient MAE 0.115728 mV
patchtst_P0_F0_pilot_60e: epoch 3/60; validation patient MAE 0.115703 mV
patchtst_P0_F0_pilot_60e: epoch 4/60; validation patient MAE 0.115681 mV
patchtst_P0_F0_pilot_60e: epoch 5/60; validation patient MAE 0.115659 mV
patchtst_P0_F0_pilot_60e: epoch 6/60; validation patient MAE 0.115638 mV
patchtst_P0_F0_pilot_60e: epoch 7/60; validation patient MAE 0.115618 mV
patchtst_P0_F0_pilot_60e: epoch 8/60; validation patient MAE 0.115599 mV
patchtst_P0_F0_pilot_60e: epoch 9/60; validation patient MAE 0.115580 mV
patchtst_P0_F0_pilot_60e: epoch 10/60; validation patient MAE 0.115560 mV
patchtst_P0_F0_pilot_60e: epoch 11/60; validation patient MAE 0.115542 mV
patchtst_P0_F0_pilot_60e: epoch 12/60; validation patient MAE 0.115526 mV
patchtst_P0_F0_pilot_60e: epoch 13/60; validation patient MAE 0.115511 mV
patchtst_P0_F0_pilot_60e: epoch 14/60; validati

In [14]:
cfg["run_mode"] = "full"

# Re-prepare data for the complete dataset split
cache, manifest = prepare(cfg)
print("Full Cache Path:", cache)
print("Full Splits:", manifest["splits"])

Full Cache Path: data/study_v3/fbc890c32f55856f
Full Splits: {'train': {'records': 17418, 'patients': 15023, 'windows': 17418}, 'val': {'records': 2183, 'patients': 1942, 'windows': 2183}}


In [12]:
from train import train_run

# Run PatchTST on the full dataset for 120 epochs
run_full_120 = train_run(
    cfg,
    cache,
    architecture="patchtst",
    preprocessing="P0",
    feature_set="F0",
    epochs=120,
    name="patchtst_P0_F0_full_120e"
)

print(f"Full 120-Epoch Training Completed! Checkpoint saved at: {run_full_120}")

patchtst_P0_F0_full_120e: epoch 1/120; validation patient MAE 0.113871 mV
patchtst_P0_F0_full_120e: epoch 2/120; validation patient MAE 0.113770 mV
patchtst_P0_F0_full_120e: epoch 3/120; validation patient MAE 0.113747 mV
patchtst_P0_F0_full_120e: epoch 4/120; validation patient MAE 0.113735 mV
patchtst_P0_F0_full_120e: epoch 5/120; validation patient MAE 0.113749 mV
patchtst_P0_F0_full_120e: epoch 6/120; validation patient MAE 0.113731 mV
patchtst_P0_F0_full_120e: epoch 7/120; validation patient MAE 0.113735 mV
patchtst_P0_F0_full_120e: epoch 8/120; validation patient MAE 0.113749 mV
patchtst_P0_F0_full_120e: epoch 9/120; validation patient MAE 0.113750 mV
patchtst_P0_F0_full_120e: epoch 10/120; validation patient MAE 0.113736 mV
patchtst_P0_F0_full_120e: epoch 11/120; validation patient MAE 0.113738 mV
patchtst_P0_F0_full_120e: epoch 12/120; validation patient MAE 0.113732 mV
patchtst_P0_F0_full_120e: epoch 13/120; validation patient MAE 0.113717 mV
patchtst_P0_F0_full_120e: epoch 14

In [18]:
cfg["run_mode"] = "full"

cache, manifest = prepare(cfg)

print("Full Cache Path:", cache)
print("Full Splits:", manifest["splits"])

Full Cache Path: data/study_v3/fbc890c32f55856f
Full Splits: {'train': {'records': 17418, 'patients': 15023, 'windows': 17418}, 'val': {'records': 2183, 'patients': 1942, 'windows': 2183}}


In [25]:
from train import train_run

run = train_run(
    cfg,
    cache,
    architecture="patchtst",
    preprocessing="P1",
    feature_set="F0",
    epochs=120,
    name="patchtst_P1_F0_full_120e",
    resume = True,
)

patchtst_P1_F0_full_120e: epoch 3/120; validation patient MAE 0.113747 mV
patchtst_P1_F0_full_120e: epoch 4/120; validation patient MAE 0.113735 mV
patchtst_P1_F0_full_120e: epoch 5/120; validation patient MAE 0.113749 mV
patchtst_P1_F0_full_120e: epoch 6/120; validation patient MAE 0.113731 mV
patchtst_P1_F0_full_120e: epoch 7/120; validation patient MAE 0.113735 mV
patchtst_P1_F0_full_120e: epoch 8/120; validation patient MAE 0.113749 mV
patchtst_P1_F0_full_120e: epoch 9/120; validation patient MAE 0.113750 mV
patchtst_P1_F0_full_120e: epoch 10/120; validation patient MAE 0.113736 mV
patchtst_P1_F0_full_120e: epoch 11/120; validation patient MAE 0.113738 mV
patchtst_P1_F0_full_120e: epoch 12/120; validation patient MAE 0.113732 mV
patchtst_P1_F0_full_120e: epoch 13/120; validation patient MAE 0.113717 mV
patchtst_P1_F0_full_120e: epoch 14/120; validation patient MAE 0.113736 mV
patchtst_P1_F0_full_120e: epoch 15/120; validation patient MAE 0.113728 mV
patchtst_P1_F0_full_120e: epoch 

In [26]:

print(f"Full 120-Epoch Training Completed! Checkpoint saved at: {run_full_120}")

Full 120-Epoch Training Completed! Checkpoint saved at: results/primary_7s3s_v3/full/checkpoints/patchtst_P0_F0_full_120e


# Implementing PatchTST with Patch size 32

In [9]:
# P0 data

from train import train_run

cfg["run_mode"] = "full"

run = train_run(
    cfg,
    cache,
    architecture="patchtst_32",
    preprocessing="P0",
    feature_set="F0",
    epochs=120,
    name="patchtst32_P0_F0_full_60e"
)

patchtst32_P0_F0_full_60e: epoch 1/120; validation patient MAE 0.115598 mV
patchtst32_P0_F0_full_60e: epoch 2/120; validation patient MAE 0.115575 mV
patchtst32_P0_F0_full_60e: epoch 3/120; validation patient MAE 0.115553 mV
patchtst32_P0_F0_full_60e: epoch 4/120; validation patient MAE 0.115533 mV
patchtst32_P0_F0_full_60e: epoch 5/120; validation patient MAE 0.115513 mV
patchtst32_P0_F0_full_60e: epoch 6/120; validation patient MAE 0.115494 mV
patchtst32_P0_F0_full_60e: epoch 7/120; validation patient MAE 0.115477 mV
patchtst32_P0_F0_full_60e: epoch 8/120; validation patient MAE 0.115460 mV
patchtst32_P0_F0_full_60e: epoch 9/120; validation patient MAE 0.115443 mV
patchtst32_P0_F0_full_60e: epoch 10/120; validation patient MAE 0.115425 mV
patchtst32_P0_F0_full_60e: epoch 11/120; validation patient MAE 0.115410 mV
patchtst32_P0_F0_full_60e: epoch 12/120; validation patient MAE 0.115395 mV
patchtst32_P0_F0_full_60e: epoch 13/120; validation patient MAE 0.115381 mV
patchtst32_P0_F0_full

In [12]:
from train import train_run

cfg["run_mode"] = "full"

run = train_run(
    cfg,
    cache,
    architecture="patchtst_32",
    preprocessing="P1",
    feature_set="F0",
    epochs=120,
    name="patchtst32_P1_F0_full_120e"
)

patchtst32_P1_F0_full_120e: epoch 1/120; validation patient MAE 0.115598 mV
patchtst32_P1_F0_full_120e: epoch 2/120; validation patient MAE 0.115575 mV
patchtst32_P1_F0_full_120e: epoch 3/120; validation patient MAE 0.115553 mV
patchtst32_P1_F0_full_120e: epoch 4/120; validation patient MAE 0.115533 mV
patchtst32_P1_F0_full_120e: epoch 5/120; validation patient MAE 0.115513 mV
patchtst32_P1_F0_full_120e: epoch 6/120; validation patient MAE 0.115494 mV
patchtst32_P1_F0_full_120e: epoch 7/120; validation patient MAE 0.115477 mV
patchtst32_P1_F0_full_120e: epoch 8/120; validation patient MAE 0.115460 mV
patchtst32_P1_F0_full_120e: epoch 9/120; validation patient MAE 0.115443 mV
patchtst32_P1_F0_full_120e: epoch 10/120; validation patient MAE 0.115425 mV
patchtst32_P1_F0_full_120e: epoch 11/120; validation patient MAE 0.115410 mV
patchtst32_P1_F0_full_120e: epoch 12/120; validation patient MAE 0.115395 mV
patchtst32_P1_F0_full_120e: epoch 13/120; validation patient MAE 0.115382 mV
patchtst

# Hyperparameter Experiment

In [22]:
cfg_large = load_config(
    PROJECT_ROOT / "configs/patchtst_large.yaml"
)

cfg_large["data"]["root"] = (
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL"
)

print(cfg_large["model"])

{'hidden': 128, 'depth': 2, 'dropout': 0.2, 'head_width': 256, 'static_embedding': 16}


In [23]:
from models import build_model

model = build_model(
    "patchtst",
    static_dim=0,
    horizon=cfg_large["data"]["horizon"],
    **cfg_large["model"]
)

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"{n_params:,}")

4,954,256


In [25]:
cfg_large["run_mode"] = "full"
cache, manifest = prepare(cfg_large)

print(manifest["splits"])

{'train': {'records': 17418, 'patients': 15023, 'windows': 17418}, 'val': {'records': 2183, 'patients': 1942, 'windows': 2183}}


In [26]:
print(cfg_large["run_mode"])

full


In [27]:
from train import train_run

train_run(
    cfg_large,
    cache,
    architecture="patchtst",
    preprocessing="P0",
    feature_set="F0",
    epochs=120,
    name="patchtst_large_P0_F0_full_120e",
)

patchtst_large_P0_F0_full_120e: epoch 1/120; validation patient MAE 0.113809 mV
patchtst_large_P0_F0_full_120e: epoch 2/120; validation patient MAE 0.113757 mV
patchtst_large_P0_F0_full_120e: epoch 3/120; validation patient MAE 0.113744 mV
patchtst_large_P0_F0_full_120e: epoch 4/120; validation patient MAE 0.113735 mV
patchtst_large_P0_F0_full_120e: epoch 5/120; validation patient MAE 0.113748 mV
patchtst_large_P0_F0_full_120e: epoch 6/120; validation patient MAE 0.113731 mV
patchtst_large_P0_F0_full_120e: epoch 7/120; validation patient MAE 0.113735 mV
patchtst_large_P0_F0_full_120e: epoch 8/120; validation patient MAE 0.113749 mV
patchtst_large_P0_F0_full_120e: epoch 9/120; validation patient MAE 0.113750 mV
patchtst_large_P0_F0_full_120e: epoch 10/120; validation patient MAE 0.113736 mV
patchtst_large_P0_F0_full_120e: epoch 11/120; validation patient MAE 0.113738 mV
patchtst_large_P0_F0_full_120e: epoch 12/120; validation patient MAE 0.113732 mV
patchtst_large_P0_F0_full_120e: epoch

PosixPath('results/primary_7s3s_v3/full/checkpoints/patchtst_large_P0_F0_full_120e')

In [28]:
from train import train_run

train_run(
    cfg_large,
    cache,
    architecture="patchtst",
    preprocessing="P1",
    feature_set="F0",
    epochs=120,
    name="patchtst_large_P1_F0_full_120e",
)

patchtst_large_P1_F0_full_120e: epoch 1/120; validation patient MAE 0.113809 mV
patchtst_large_P1_F0_full_120e: epoch 2/120; validation patient MAE 0.113757 mV
patchtst_large_P1_F0_full_120e: epoch 3/120; validation patient MAE 0.113744 mV
patchtst_large_P1_F0_full_120e: epoch 4/120; validation patient MAE 0.113735 mV
patchtst_large_P1_F0_full_120e: epoch 5/120; validation patient MAE 0.113748 mV
patchtst_large_P1_F0_full_120e: epoch 6/120; validation patient MAE 0.113731 mV
patchtst_large_P1_F0_full_120e: epoch 7/120; validation patient MAE 0.113735 mV
patchtst_large_P1_F0_full_120e: epoch 8/120; validation patient MAE 0.113749 mV
patchtst_large_P1_F0_full_120e: epoch 9/120; validation patient MAE 0.113750 mV
patchtst_large_P1_F0_full_120e: epoch 10/120; validation patient MAE 0.113736 mV
patchtst_large_P1_F0_full_120e: epoch 11/120; validation patient MAE 0.113738 mV
patchtst_large_P1_F0_full_120e: epoch 12/120; validation patient MAE 0.113732 mV
patchtst_large_P1_F0_full_120e: epoch

PosixPath('results/primary_7s3s_v3/full/checkpoints/patchtst_large_P1_F0_full_120e')